In [ ]:
#imports

#webdriver imports
#used selenium 3.0.2
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time

#download and trasncribe imports
#used ty_dlp 2024.10.22
import yt_dlp
#used faster whisper 1.1.0
from faster_whisper import WhisperModel
import os
import pandas as pd
import json



In [ ]:
#choose church page - this is the only input to the whole process
url="https://www.facebook.com/GrandRapidsFirst/live_videos"
church_name='Grand_Rapids_First'

In [ ]:
#log into facebook for the url collection - most videos require login
driver = webdriver.Chrome(executable_path="~filepath\\chromedriver.exe")
driver.get(url)

In [ ]:
#get urls for videos at url link
def get_page_html(url):
    driver.get(url)
    repeat=0
    #This repeat number could be more or less depending on how far back you want to do
    while repeat<75:
        #wait times are here to ensure chromedriver has time to load the page after scrolling
        driver.implicitly_wait(4000)
        page_html = driver.page_source
        driver.execute_script("window.scrollBy(0, 1000000);")
        driver.implicitly_wait(4000)
        repeat+=1
    page_html2 = driver.page_source
    return page_html2

html_content = get_page_html(url)

driver.quit()

#get urls from the html
list1=[]
url_list=[]
for x in html_content.split("/videos/"):
    list1.append(x.split('"')[0])
for x in list1:
    if len(x)>11:
        if x in url_list:
            pass
        else:
            url_list.append(x)

#sometimes this doesn't work. If it does, you will get a good number of videos found
print(str(len(url_list))+' videos found')
        

In [ ]:
#this cell is only needed if the scraping in the cell above didn't work
#copy the entire html for the link and paste it in the string in this cell
html_2=""" insert html here """


In [ ]:
#this grabs the urls from the html 
html_2=html_2.replace("\n"," ")
list1=[]
url_list=[]
for x in html_2.split("/videos/"):
    list1.append(x.split('"')[0])
for x in list1:
    if len(x)>11:
        if x in url_list:
            pass
        else:
            url_list.append(x)
#the first result will not be a real video from the page and can be omitted
url_list=url_list[1:]
#if this still doesn't produce the expected number of videos, find another way to build the url list
print(str(len(url_list))+' videos found')

In [ ]:
#download and transcribe the urls
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'
#Can use the large model if your future queries will depend on high accuracy transcription
#The queries used in this example don't depend on individual words, hence the medium model
model_size = "medium.en"

#load whisper model
#the device being set to cuda run the model on a graphics card, could also be done on cpu
model = WhisperModel(model_size, device="cuda", compute_type="float16")

#set the prev_date to a day in the future
#set the stop_date to a day when you stop collecting videos - older videos are ignored
prev_date='20260101'
stop_date='20250101'

failed_videos=[]

for v in url_list:
    if prev_date<stop_date:
        print('reached stop date')
    elif os.path.exists("~filepath\\"+church_name+'\\'+str(v)+"_transcribe.csv"):
        print('already downloaded '+str(v))
    else:
        #the actual url downloaded comes from m.facebook, the mobile site
        video_url = 'https://m.facebook.com/watch/live/?v='+str(v)

        # yt-dlp options for audio extraction
        ydl_opts = {
            #you need an actual facebook username and password here to get most videos
            'username':'username here',
            'password':'password here',
            'format': 'bestaudio/best',
            'writeinfojson': True,
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'wav',  
            }],
            #this is the folder where .wav audio files go (they get deleted soon)
            #this is also where the metadata json is sent 
            'outtmpl': '~filepath\\'+church_name+"\\"+'audio_'+str(v)+'.wav',
        }



        # Download the audio file
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([video_url])
        except:
            if os.path.exists("~filepath\\"+church_name+"\\audio_"+str(v)+'.wav'):
                print('downloaded '+str(v))
                segments, info = model.transcribe("~filepath\\"+church_name+"\\audio_"+str(v)+'.wav', beam_size=5)
                #whisper works with multiple languages (so do LLMs)
                print("Detected language '%s' with probability %f" % (info.language, info.language_probability))
                output=[]
                for segment in segments:
                    a="[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text)
                    output.append(a)
                a=pd.DataFrame()
                a['Text']=output
                #this csv file is the transcript of the service
                a.to_csv("~filepath\\"+church_name+'\\'+str(v)+"_transcribe.csv")
                #the .wav file can be deleted once you have the transcript
                os.remove("~filepath\\"+church_name+"\\audio_"+str(v)+'.wav')
            else:
                print('no download? '+str(v))
        if os.path.exists("~filepath\\"+church_name+'\\'+str(v)+"_transcribe.csv"):
            try:
                #this opens the metadata json and grabs the date
                #if the stop date has been reached, no more videos download
                with open('~filepath\\'+church_name+'\\audio_'+str(v)+'.wav.info.json', encoding='utf-8') as fh:
                    data = json.load(fh)
                prev_date=data['upload_date']
                print('')
                print(prev_date)
                print('')
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")
        else:
            failed_videos.append(v)
            a=pd.DataFrame()
            a['failed_videos']=failed_videos
            #this failed videos list could be useful since some videos will work if you try again later
            a.to_csv('~filepath\\'+church_name+"\\failed_videos.csv")
